In [8]:
from lib.normalizer import min_max_only_ppi
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
import pickle

In [9]:
# load the plat_cleaned_input.csv
df = pd.read_csv('input/plat_cleaned_input.csv', index_col=0)
# load the ORvsCRS df
orvscrs_df = pd.read_csv('input/orvscrs_reformatted_df.csv', index_col=0)
# load the control
control_df = pd.read_csv('input/pure_or_0.25_thresh_paired_df.csv', index_col=0)

display(df)
display(orvscrs_df)
display(control_df)

,2A_BIRC3,BIRC3_BIRC3,FYB_BIRC3,LCK_BIRC3,NCK_BIRC3,PKCt_BIRC3,SLP76_BIRC3,TAK1_BIRC3,TCRb_BIRC3,TRAF1_BIRC3,...,cd3,type,control,has control,poor response inverse of remission,CRS,NT,optimal response,response type,stim type
CAR_19_447,94.697900,159.000467,111.025340,95.692668,102.449484,116.653537,109.266332,110.618228,105.406606,59.761530,...,0.0,3.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,19
CAR_19_464,105.715935,100.395341,95.103711,106.064396,95.958365,97.413419,96.677088,100.854728,97.338542,81.353148,...,0.0,3.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,19
CAR_19_464_1,106.692161,75.011666,93.534483,106.956522,97.537879,93.095238,99.365751,104.187192,93.555094,72.747554,...,0.0,3.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,19
CAR_19_464_2,106.009615,110.487201,92.600897,108.830022,91.312741,98.976982,94.262295,99.270073,98.242188,102.555629,...,0.0,3.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,19
CAR_19_467,86.282699,95.343400,93.226090,90.969214,106.721199,97.303476,108.256138,109.724197,92.509792,102.080979,...,0.0,3.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CAR_N3_846,106.860822,99.910535,102.576999,100.000000,98.061693,94.891145,107.306833,96.088469,93.687998,97.869716,...,1.0,2.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,N3
CAR_N3_852,103.686396,107.676357,100.363567,99.722339,91.279267,112.796040,101.209913,101.389286,101.748295,97.289664,...,1.0,2.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,N3
CAR_N3_886,97.267920,99.755648,102.180294,96.130964,107.304212,108.077007,98.296568,106.537091,95.087806,99.258598,...,1.0,2.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,N3
CAR_N3_892,109.773411,97.928661,101.206907,92.493674,93.287443,93.713144,106.295268,103.261280,91.170382,106.746778,...,1.0,2.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,N3


,2A_ZAP70,2A_FYN,2A_CD3z,2A_LCK,2A_PKCt,2A_SHP2,2A_CBL-b,2A_GRAP2,2A_CD28,2A_TRAF1,...,UBASH3A_UBASH3A,experiment,stim,CRS,or,thaw,car%,protein conc,prev index,pair num
OR_US_1,171.977271,1326.826370,27890.286491,225.533885,407.623917,3853.951461,329.027874,151.282773,269.926369,241.518640,...,1728.286894,3,0,0,1,2,88.67,2.6,OR_US_1,U1
OR_US_2,152.036036,2333.413064,24870.577672,212.498949,398.975130,3742.537687,314.141433,145.566933,237.985322,253.984965,...,1875.685186,2,0,0,1,1,57.90,2.0,OR_US_2,U2
OR_US_3,171.833939,2645.007898,24601.651507,216.240271,450.513204,3500.470500,347.401016,150.564545,296.333253,333.590474,...,1684.103314,1,0,0,1,1,90.34,1.9,OR_US_3,U3
OR_US_4,184.510399,1568.404442,27480.535061,235.219403,524.445539,4410.044140,388.095105,146.552843,295.954319,294.391169,...,1755.340287,4,0,0,1,2,72.40,2.0,OR_US_4,U4
OR_US_5,173.910005,1218.177619,26659.095137,214.261732,356.773175,2894.992007,259.721716,152.671717,225.583882,239.451809,...,1832.742001,5,0,0,1,2,29.44,2.0,OR_US_5,U5
CRS_US_1,172.985378,1241.610086,27992.529925,211.427803,381.363260,3348.688284,299.659927,150.738861,267.562208,278.308620,...,1850.461008,3,0,1,0,2,47.83,2.6,CRS_US_1,U1
CRS_US_2,162.705279,1564.576195,24418.733329,202.847959,402.068277,3232.755270,313.920647,148.882987,246.147866,257.704937,...,1824.185576,2,0,1,0,1,97.89,2.0,CRS_US_2,U2
CRS_US_3,174.266170,2362.884665,24671.408173,213.609408,399.963829,3177.998633,309.345532,141.207909,243.279101,238.910114,...,1917.611182,1,0,1,0,1,33.00,1.9,CRS_US_3,U3
CRS_US_4,171.377740,1433.955555,27242.152794,224.213553,438.464851,3688.360057,344.875666,153.131227,251.232299,265.120878,...,1933.952380,4,0,1,0,2,52.67,2.0,CRS_US_4,U4
CRS_US_5,174.206271,1465.154718,28225.054087,230.207072,451.493101,4059.731032,361.028272,151.075846,305.541690,346.823479,...,1964.914230,5,0,1,0,2,60.54,2.0,CRS_US_5,U5


,2A_BIRC3,BIRC3_BIRC3,FYB_BIRC3,LCK_BIRC3,NCK_BIRC3,PKCt_BIRC3,SLP76_BIRC3,TAK1_BIRC3,TCRb_BIRC3,TRAF1_BIRC3,...,type,control,has control,poor response inverse of remission,CRS,NT,optimal response,response type,stim type,pair num
CAR_KP_846,104.390926,93.379602,92.412169,95.943745,102.369042,101.021771,98.892904,103.293921,94.289141,94.567775,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,KP,9011
CAR_KP_803,108.076928,100.000000,95.681090,101.714550,102.394124,103.831909,107.186530,103.830163,94.073516,88.771979,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,KP,9011
CAR_KP_886,101.707550,97.678659,108.721176,104.324216,102.213397,101.121806,97.274509,111.766763,102.311621,102.817327,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,KP,9012
CAR_KP_894,86.209949,96.910921,95.950930,96.836800,102.487153,89.028886,99.027030,98.938543,105.909372,110.657330,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,KP,9012
CAR_KP_452,92.804604,95.070947,95.871389,95.218166,101.585217,106.054641,112.998149,96.569840,98.503646,99.351938,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,KP,9009
CAR_KP_494,100.000000,101.974305,104.700786,102.875687,103.785790,103.309914,99.165727,102.474005,97.976262,94.188272,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,KP,9009
CAR_KP_486,103.372065,99.118577,102.430141,105.725184,103.183604,100.000000,103.231987,97.224938,102.997071,104.727663,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,KP,9010
CAR_KP_411,103.966982,101.153596,106.111432,108.651310,103.970042,105.535455,103.051980,100.605559,101.620527,94.749102,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,KP,9010
CAR_N3_892,109.773411,97.928661,101.206907,92.493674,93.287443,93.713144,106.295268,103.261280,91.170382,106.746778,...,2.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,N3,9007
CAR_N3_846,106.860822,99.910535,102.576999,100.000000,98.061693,94.891145,107.306833,96.088469,93.687998,97.869716,...,2.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,N3,9007


In [10]:
# Create list of columns to potentially drop
columns_to_drop = [
    'cd3', 'type', 'control', 'has control',
    'poor response inverse of remission', 'NT', 'optimal response',
    'response type', 'patient', 'experiment', 'cell count',
    '% CAR', 'thaw', 'stim', 'cd19'  # Note: 'cd3' was duplicated in original
]

# Drop what exists, ignore what doesn't
df = df.drop(columns=columns_to_drop, errors='ignore')

# Optional: See which columns were actually dropped
print(f"Columns dropped: {set(columns_to_drop) - set(df.columns)}")
df

Columns dropped: {'% CAR', 'type', 'experiment', 'NT', 'cd3', 'response type', 'cell count', 'has control', 'cd19', 'optimal response', 'poor response inverse of remission', 'control', 'patient', 'thaw', 'stim'}


,2A_BIRC3,BIRC3_BIRC3,FYB_BIRC3,LCK_BIRC3,NCK_BIRC3,PKCt_BIRC3,SLP76_BIRC3,TAK1_BIRC3,TCRb_BIRC3,TRAF1_BIRC3,...,TAK1_ZAP70,TCRb_ZAP70,TRAF1_ZAP70,TRAF2_ZAP70,VAV1_ZAP70,ZAP70_ZAP70,cd4v8,protein conc,CRS,stim type
CAR_19_447,94.697900,159.000467,111.025340,95.692668,102.449484,116.653537,109.266332,110.618228,105.406606,59.761530,...,95.670225,92.892627,104.046637,100.633391,89.477148,61.399937,4.0,1.740,1.0,19
CAR_19_464,105.715935,100.395341,95.103711,106.064396,95.958365,97.413419,96.677088,100.854728,97.338542,81.353148,...,98.671928,101.229067,103.222083,95.524683,104.400218,68.545724,4.0,2.230,1.0,19
CAR_19_464_1,106.692161,75.011666,93.534483,106.956522,97.537879,93.095238,99.365751,104.187192,93.555094,72.747554,...,94.320988,102.000000,102.412869,96.194503,134.579439,58.124373,4.0,2.230,1.0,19
CAR_19_464_2,106.009615,110.487201,92.600897,108.830022,91.312741,98.976982,94.262295,99.270073,98.242188,102.555629,...,100.246305,101.072961,106.779661,89.086860,96.259352,56.084337,4.0,2.210,1.0,19
CAR_19_467,86.282699,95.343400,93.226090,90.969214,106.721199,97.303476,108.256138,109.724197,92.509792,102.080979,...,100.979144,100.721816,94.883577,96.859977,101.986777,46.855297,4.0,2.680,1.0,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CAR_N3_846,106.860822,99.910535,102.576999,100.000000,98.061693,94.891145,107.306833,96.088469,93.687998,97.869716,...,96.973599,93.387824,96.489995,97.877504,98.559626,60.844700,8.0,3.049,0.0,N3
CAR_N3_852,103.686396,107.676357,100.363567,99.722339,91.279267,112.796040,101.209913,101.389286,101.748295,97.289664,...,99.246506,96.320792,100.000000,108.064257,104.876472,96.276297,8.0,1.767,0.0,N3
CAR_N3_886,97.267920,99.755648,102.180294,96.130964,107.304212,108.077007,98.296568,106.537091,95.087806,99.258598,...,96.536627,106.216287,97.007796,98.830243,98.375225,120.770105,8.0,2.400,0.0,N3
CAR_N3_892,109.773411,97.928661,101.206907,92.493674,93.287443,93.713144,106.295268,103.261280,91.170382,106.746778,...,99.588932,100.893874,94.997704,100.000000,95.424008,66.970048,8.0,3.180,0.0,N3


In [11]:
# add a column pair name to the df
df['pair num'] = None

# add the columns that are needed
orvscrs_df['stim type'] = None
orvscrs_df['cd4v8'] = 4

df

,2A_BIRC3,BIRC3_BIRC3,FYB_BIRC3,LCK_BIRC3,NCK_BIRC3,PKCt_BIRC3,SLP76_BIRC3,TAK1_BIRC3,TCRb_BIRC3,TRAF1_BIRC3,...,TCRb_ZAP70,TRAF1_ZAP70,TRAF2_ZAP70,VAV1_ZAP70,ZAP70_ZAP70,cd4v8,protein conc,CRS,stim type,pair num
CAR_19_447,94.697900,159.000467,111.025340,95.692668,102.449484,116.653537,109.266332,110.618228,105.406606,59.761530,...,92.892627,104.046637,100.633391,89.477148,61.399937,4.0,1.740,1.0,19,None
CAR_19_464,105.715935,100.395341,95.103711,106.064396,95.958365,97.413419,96.677088,100.854728,97.338542,81.353148,...,101.229067,103.222083,95.524683,104.400218,68.545724,4.0,2.230,1.0,19,None
CAR_19_464_1,106.692161,75.011666,93.534483,106.956522,97.537879,93.095238,99.365751,104.187192,93.555094,72.747554,...,102.000000,102.412869,96.194503,134.579439,58.124373,4.0,2.230,1.0,19,None
CAR_19_464_2,106.009615,110.487201,92.600897,108.830022,91.312741,98.976982,94.262295,99.270073,98.242188,102.555629,...,101.072961,106.779661,89.086860,96.259352,56.084337,4.0,2.210,1.0,19,None
CAR_19_467,86.282699,95.343400,93.226090,90.969214,106.721199,97.303476,108.256138,109.724197,92.509792,102.080979,...,100.721816,94.883577,96.859977,101.986777,46.855297,4.0,2.680,1.0,19,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CAR_N3_846,106.860822,99.910535,102.576999,100.000000,98.061693,94.891145,107.306833,96.088469,93.687998,97.869716,...,93.387824,96.489995,97.877504,98.559626,60.844700,8.0,3.049,0.0,N3,None
CAR_N3_852,103.686396,107.676357,100.363567,99.722339,91.279267,112.796040,101.209913,101.389286,101.748295,97.289664,...,96.320792,100.000000,108.064257,104.876472,96.276297,8.0,1.767,0.0,N3,None
CAR_N3_886,97.267920,99.755648,102.180294,96.130964,107.304212,108.077007,98.296568,106.537091,95.087806,99.258598,...,106.216287,97.007796,98.830243,98.375225,120.770105,8.0,2.400,0.0,N3,None
CAR_N3_892,109.773411,97.928661,101.206907,92.493674,93.287443,93.713144,106.295268,103.261280,91.170382,106.746778,...,100.893874,94.997704,100.000000,95.424008,66.970048,8.0,3.180,0.0,N3,None


In [12]:
# Find columns that exist in all dataframes
common_columns = [col for col in df.columns if col in orvscrs_df.columns and col in control_df.columns]
print(common_columns)
# Select only common columns
df = df[common_columns]
orvscrs_df = orvscrs_df[common_columns]
control_df = control_df[common_columns]

display(df)
display(orvscrs_df)
control_df

['2A_BIRC3', 'BIRC3_BIRC3', 'FYB_BIRC3', 'LCK_BIRC3', 'NCK_BIRC3', 'PKCt_BIRC3', 'TRAF1_BIRC3', 'TRAF2_BIRC3', 'VAV1_BIRC3', '2A_CD28', 'BIRC3_CD28', 'CD28_CD28', 'FYB_CD28', 'LCK_CD28', 'NCK_CD28', 'PI3K_CD28', 'PKCt_CD28', 'SLP76_CD28', 'TRAF1_CD28', 'TRAF2_CD28', '2A_CD3z', 'BIRC3_CD3z', 'CD28_CD3z', 'FYB_CD3z', 'LCK_CD3z', 'NCK_CD3z', 'PI3K_CD3z', 'PKCt_CD3z', 'SLP76_CD3z', 'TCRb_CD3z', 'TRAF1_CD3z', 'TRAF2_CD3z', 'VAV1_CD3z', '2A_FYN', 'BIRC3_FYN', 'CD28_FYN', 'FYB_FYN', 'LCK_FYN', 'NCK_FYN', 'PI3K_FYN', 'PKCt_FYN', 'SLP76_FYN', 'TAK1_FYN', 'TCRb_FYN', 'TRAF1_FYN', 'TRAF2_FYN', 'VAV1_FYN', 'ZAP70_FYN', '2A_GRAP2', 'BIRC3_GRAP2', 'CD28_GRAP2', 'FYB_GRAP2', 'LCK_GRAP2', 'NCK_GRAP2', 'PI3K_GRAP2', 'PKCt_GRAP2', 'SLP76_GRAP2', 'TRAF1_GRAP2', 'TRAF2_GRAP2', 'VAV1_GRAP2', '2A_LAT', 'BIRC3_LAT', 'CD28_LAT', 'FYB_LAT', 'LAT_LAT', 'LCK_LAT', 'NCK_LAT', 'PI3K_LAT', 'PKCt_LAT', 'SLP76_LAT', 'TCRb_LAT', 'TRAF1_LAT', 'TRAF2_LAT', 'VAV1_LAT', '2A_LCK', 'BIRC3_LCK', 'CD28_LCK', 'FYB_LCK', 'LCK_L

,2A_BIRC3,BIRC3_BIRC3,FYB_BIRC3,LCK_BIRC3,NCK_BIRC3,PKCt_BIRC3,TRAF1_BIRC3,TRAF2_BIRC3,VAV1_BIRC3,2A_CD28,...,PKCt_ZAP70,TRAF1_ZAP70,TRAF2_ZAP70,VAV1_ZAP70,ZAP70_ZAP70,cd4v8,protein conc,CRS,stim type,pair num
CAR_19_447,94.697900,159.000467,111.025340,95.692668,102.449484,116.653537,59.761530,91.384325,119.192217,94.043909,...,104.876816,104.046637,100.633391,89.477148,61.399937,4.0,1.740,1.0,19,None
CAR_19_464,105.715935,100.395341,95.103711,106.064396,95.958365,97.413419,81.353148,99.321770,106.735452,98.773677,...,95.549554,103.222083,95.524683,104.400218,68.545724,4.0,2.230,1.0,19,None
CAR_19_464_1,106.692161,75.011666,93.534483,106.956522,97.537879,93.095238,72.747554,104.911591,92.900609,91.084337,...,95.238095,102.412869,96.194503,134.579439,58.124373,4.0,2.230,1.0,19,None
CAR_19_464_2,106.009615,110.487201,92.600897,108.830022,91.312741,98.976982,102.555629,96.237624,132.124352,105.475040,...,93.606138,106.779661,89.086860,96.259352,56.084337,4.0,2.210,1.0,19,None
CAR_19_467,86.282699,95.343400,93.226090,90.969214,106.721199,97.303476,102.080979,100.673954,102.839009,80.721294,...,97.416123,94.883577,96.859977,101.986777,46.855297,4.0,2.680,1.0,19,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CAR_N3_846,106.860822,99.910535,102.576999,100.000000,98.061693,94.891145,97.869716,98.418604,99.158252,90.532907,...,89.504757,96.489995,97.877504,98.559626,60.844700,8.0,3.049,0.0,N3,None
CAR_N3_852,103.686396,107.676357,100.363567,99.722339,91.279267,112.796040,97.289664,95.703239,93.558571,96.715353,...,94.078226,100.000000,108.064257,104.876472,96.276297,8.0,1.767,0.0,N3,None
CAR_N3_886,97.267920,99.755648,102.180294,96.130964,107.304212,108.077007,99.258598,104.808305,83.204627,98.497445,...,103.973137,97.007796,98.830243,98.375225,120.770105,8.0,2.400,0.0,N3,None
CAR_N3_892,109.773411,97.928661,101.206907,92.493674,93.287443,93.713144,106.746778,98.844857,98.207932,92.073505,...,98.597808,94.997704,100.000000,95.424008,66.970048,8.0,3.180,0.0,N3,None


,2A_BIRC3,BIRC3_BIRC3,FYB_BIRC3,LCK_BIRC3,NCK_BIRC3,PKCt_BIRC3,TRAF1_BIRC3,TRAF2_BIRC3,VAV1_BIRC3,2A_CD28,...,PKCt_ZAP70,TRAF1_ZAP70,TRAF2_ZAP70,VAV1_ZAP70,ZAP70_ZAP70,cd4v8,protein conc,CRS,stim type,pair num
OR_US_1,140.123513,104.309708,147.401096,122.071515,112.237066,106.985702,104.596598,140.555680,126.851136,269.926369,...,112.080429,104.155719,148.789332,128.213229,1074.678465,4,2.6,0,None,U1
OR_US_2,139.338783,100.038927,151.050314,127.692394,106.418065,110.522039,100.744792,148.240737,132.103948,237.985322,...,113.319249,105.820906,148.253493,130.523874,955.714456,4,2.0,0,None,U2
OR_US_3,142.987793,107.417446,153.231292,120.435940,113.825218,105.319573,106.120020,141.503082,119.676657,296.333253,...,110.994419,104.032952,150.507823,130.689419,904.081385,4,1.9,0,None,U3
OR_US_4,140.782992,102.021659,152.207431,131.686742,113.264489,111.696622,100.561761,149.255105,125.891945,295.954319,...,106.829265,106.825000,151.341511,124.775217,809.776856,4,2.0,0,None,U4
OR_US_5,143.843327,96.722079,150.495478,127.142897,106.876936,107.913904,108.834624,149.506815,119.678488,225.583882,...,104.991574,103.200553,152.950116,125.980493,850.486981,4,2.0,0,None,U5
CRS_US_1,138.513352,93.858233,161.919904,131.326481,107.688259,104.032468,102.384353,151.679921,120.051235,267.562208,...,111.158422,104.601164,152.717379,126.353685,736.174354,4,2.6,1,None,U1
CRS_US_2,142.220038,106.586129,144.737106,119.946555,112.562186,110.522039,103.933472,142.460152,121.866658,246.147866,...,105.625768,101.539991,151.818909,128.777524,807.466242,4,2.0,1,None,U2
CRS_US_3,142.114448,96.917902,154.012106,129.609289,110.685586,104.849788,108.683718,146.638689,144.916605,243.279101,...,106.112435,104.805727,147.544791,127.617223,766.422604,4,1.9,1,None,U3
CRS_US_4,141.828298,99.152734,150.285262,131.132467,111.986445,105.506861,103.556469,152.988290,127.993343,251.232299,...,107.148096,103.853683,148.126490,129.896037,1018.430540,4,2.0,1,None,U4
CRS_US_5,149.884563,103.813868,160.556735,123.126581,113.422447,105.855565,102.442332,149.193192,134.829172,305.541690,...,105.937422,106.307165,152.146758,128.049208,817.218125,4,2.0,1,None,U5


,2A_BIRC3,BIRC3_BIRC3,FYB_BIRC3,LCK_BIRC3,NCK_BIRC3,PKCt_BIRC3,TRAF1_BIRC3,TRAF2_BIRC3,VAV1_BIRC3,2A_CD28,...,PKCt_ZAP70,TRAF1_ZAP70,TRAF2_ZAP70,VAV1_ZAP70,ZAP70_ZAP70,cd4v8,protein conc,CRS,stim type,pair num
CAR_KP_846,104.390926,93.379602,92.412169,95.943745,102.369042,101.021771,94.567775,94.860462,88.383882,87.934098,...,97.473367,94.088412,99.183655,104.561184,60.497168,8.0,3.929,0.0,KP,9011
CAR_KP_803,108.076928,100.000000,95.681090,101.714550,102.394124,103.831909,88.771979,96.054327,85.314928,94.245220,...,105.665275,103.369370,104.033786,111.781146,111.760249,8.0,3.643,0.0,KP,9011
CAR_KP_886,101.707550,97.678659,108.721176,104.324216,102.213397,101.121806,102.817327,110.475236,98.950289,86.691658,...,105.779109,94.015593,101.169757,105.849190,73.710567,8.0,2.400,0.0,KP,9012
CAR_KP_894,86.209949,96.910921,95.950930,96.836800,102.487153,89.028886,110.657330,101.528568,84.970719,94.339273,...,107.988875,93.477098,102.054943,87.014067,69.971752,8.0,2.200,0.0,KP,9012
CAR_KP_452,92.804604,95.070947,95.871389,95.218166,101.585217,106.054641,99.351938,101.258301,89.245058,126.553499,...,101.115274,102.705266,105.493406,97.553996,79.098706,4.0,1.680,0.0,KP,9009
CAR_KP_494,100.000000,101.974305,104.700786,102.875687,103.785790,103.309914,94.188272,101.189910,90.391769,96.896371,...,91.269275,101.039936,101.635752,91.746985,50.488250,4.0,1.500,0.0,KP,9009
CAR_KP_486,103.372065,99.118577,102.430141,105.725184,103.183604,100.000000,104.727663,97.554034,112.819866,124.032266,...,103.166556,97.442997,99.556699,92.935352,107.562643,4.0,3.700,0.0,KP,9010
CAR_KP_411,103.966982,101.153596,106.111432,108.651310,103.970042,105.535455,94.749102,100.986993,86.237305,116.680049,...,104.706777,96.590967,100.912866,94.373562,50.853825,4.0,3.500,0.0,KP,9010
CAR_N3_892,109.773411,97.928661,101.206907,92.493674,93.287443,93.713144,106.746778,98.844857,98.207932,92.073505,...,98.597808,94.997704,100.000000,95.424008,66.970048,8.0,3.180,0.0,N3,9007
CAR_N3_846,106.860822,99.910535,102.576999,100.000000,98.061693,94.891145,97.869716,98.418604,99.158252,90.532907,...,89.504757,96.489995,97.877504,98.559626,60.844700,8.0,3.049,0.0,N3,9007


In [13]:
# merge the ORvsCRS df with the df
df = pd.concat([df, orvscrs_df], axis=0)
df['control'] = 0

# add control_ to the front of the index for the control df and add the control column
control_df.index = ['control_' + str(i) for i in control_df.index]
control_df['control'] = 1

# merge the control df with the df
df = pd.concat([df, control_df], axis=0)
# saved as cleaned.csv
df.to_csv('input/cleaned.csv')